In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import logging
from pathlib import Path
from typing import Dict, Tuple

### Configurações iniciais de loggings e estilo visual

In [7]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Configuração de estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)

### Leitura de arquivos .csv para dados

In [ ]:
def load_data(data_dir: Path) -> Dict[str, pd.DataFrame]:
    """
    Carrega os arquivos CSV do diretório especificado.
    Adiciona tratamento de erros caso o arquivo não seja encontrado.
    """
    files = {
        'prices': 'chip_prices.csv',
        'controls': 'export_controls.csv',
        'fab': 'fab_capacity.csv',
        'ai_market': 'ai_chip_market.csv',
        'financials': 'chip_companies_financials.csv'
    }
    
    datasets = {}
    for key, filename in files.items():
        filepath = data_dir / filename
        try:
            datasets[key] = pd.read_csv(filepath)
            logging.info(f"Arquivo carregado com sucesso: {filename}")
        except FileNotFoundError:
            logging.error(f"Arquivo não encontrado: {filepath}. Verifique o diretório.")
            raise
        except Exception as e:
            logging.error(f"Erro ao carregar o arquivo {filename}: {e}")
            raise
            
    return datasets


### Préprocessamento de dados

In [9]:
def preprocess_data(datasets: Dict[str, pd.DataFrame]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Realiza a limpeza e transformação dos dados (Tratamento de nulos, filtros e agrupamentos).
    """
    logging.info("Iniciando pré-processamento dos dados...")
    
    # 1. Mercado de Chips de IA
    df_ai = datasets['ai_market'].copy()
    df_ai.dropna(subset=['estimated_revenue_usd_m', 'estimated_shipments_units'], inplace=True)
    ai_revenue_by_vendor = df_ai.groupby(['year', 'vendor'])['estimated_revenue_usd_m'].sum().reset_index()

    # 2. Preços de GPUs
    df_prices = datasets['prices'].copy()
    df_gpu_prices = df_prices[df_prices['product'].str.contains('NVIDIA|AMD|GPU', case=False, na=False)].copy()
    df_gpu_prices['year_month'] = pd.to_datetime(df_gpu_prices['year_month'])

    # 3. Financeiro (Foco: NVIDIA, AMD, TSMC)
    df_fin = datasets['financials'].copy()
    target_companies = ['NVIDIA', 'AMD', 'TSMC']
    df_fin_gpus = df_fin[df_fin['company_name'].isin(target_companies)].copy()
    df_fin_gpus.fillna(0, inplace=True)

    # 4. Capacidade Fabril (TSMC - Nós de ponta)
    df_fab = datasets['fab'].copy()
    df_fab_leading = df_fab[(df_fab['company'] == 'TSMC') & (df_fab['fab_type'] == 'logic_leading')]
    fab_capacity_yearly = df_fab_leading.groupby('year')['monthly_wafer_capacity'].sum().reset_index()

    # 5. Geopolítica (Sanções Graves)
    df_controls = datasets['controls'].copy()
    df_controls_high_severity = df_controls[df_controls['severity_score'] >= 7]
    sanctions_per_year = df_controls_high_severity.groupby('year')['control_id'].count().reset_index()
    sanctions_per_year.rename(columns={'control_id': 'num_sancoes_graves'}, inplace=True)
    
    logging.info("Pré-processamento concluído.")
    return df_ai, ai_revenue_by_vendor, df_gpu_prices, df_fin_gpus, fab_capacity_yearly, sanctions_per_year

### Funções de plots

In [10]:
def plot_dashboards(ai_rev: pd.DataFrame, df_fin: pd.DataFrame, df_gpu_prices: pd.DataFrame, fab_cap: pd.DataFrame, sanctions: pd.DataFrame):
    """
    Gera o painel de visualização com matplotlib e seaborn.
    """
    logging.info("Gerando visualizações...")
    fig, axs = plt.subplots(2, 2)
    fig.suptitle('Análise Exploratória: Mercado Global de GPUs e Chips de IA', fontsize=18, fontweight='bold')

    # Gráfico 1: Evolução da Receita
    for vendor in ai_rev['vendor'].unique():
        subset = ai_rev[ai_rev['vendor'] == vendor]
        axs[0, 0].plot(subset['year'], subset['estimated_revenue_usd_m'], marker='o', label=vendor)
    axs[0, 0].set_title('Receita Estimada de Chips de IA por Empresa (USD m)')
    axs[0, 0].set_ylabel('Receita (USD m)')
    axs[0, 0].legend()

    # Gráfico 2: P&D vs Receita Total
    df_fin_subset = df_fin[df_fin['company_name'].isin(['NVIDIA', 'AMD'])]
    sns.scatterplot(data=df_fin_subset, x='rd_spend_usd_bn', y='revenue_usd_bn', 
                    hue='company_name', size='year', sizes=(50, 200), ax=axs[0, 1])
    axs[0, 1].set_title('Gastos em P&D vs Receita Total (Bilhões USD)')
    axs[0, 1].set_xlabel('Gastos em P&D (USD bn)')
    axs[0, 1].set_ylabel('Receita Total (USD bn)')

    # Gráfico 3: Preço da H100
    h100_prices = df_gpu_prices[df_gpu_prices['product'].str.contains('H100', na=False)]
    if not h100_prices.empty:
        axs[1, 0].plot(h100_prices['year_month'], h100_prices['price'], color='purple', linewidth=2)
        axs[1, 0].set_title('Evolução do Preço Unitário: NVIDIA H100')
        axs[1, 0].set_ylabel('Preço (USD)')
        axs[1, 0].tick_params(axis='x', rotation=45)
    else:
        axs[1, 0].text(0.5, 0.5, 'Dados da H100 indisponíveis', ha='center', va='center')

    # Gráfico 4: Capacidade vs Sanções
    ax2 = axs[1, 1].twinx()
    axs[1, 1].bar(fab_cap['year'], fab_cap['monthly_wafer_capacity'], color='skyblue', label='Capacidade TSMC', alpha=0.7)
    ax2.plot(sanctions['year'], sanctions['num_sancoes_graves'], color='red', marker='x', linewidth=2, label='Sanções Graves')
    axs[1, 1].set_title('Capacidade de Produção TSMC vs Sanções Globais')
    axs[1, 1].set_ylabel('Capacidade (Wafers/mês)')
    ax2.set_ylabel('Nº de Sanções (Score >= 7)')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

def generate_descriptive_stats(df_ai: pd.DataFrame, df_fin: pd.DataFrame):
    """
    Exibe estatísticas descritivas formatadas no console/log.
    """
    logging.info("Calculando estatísticas descritivas...")
    print("\n" + "="*50)
    print("ESTATÍSTICAS DESCRITIVAS: CHIPS DE IA")
    print("="*50)
    print(df_ai[['estimated_asp_usd', 'estimated_revenue_usd_m']].describe().round(2))
    
    print("\n" + "="*50)
    print("RECEITA MÉDIA (USD bn) POR EMPRESA")
    print("="*50)
    print(df_fin.groupby('company_name')['revenue_usd_bn'].mean().round(2).sort_values(ascending=False))

### Execução principal do código

In [11]:
def main():
    # Define o diretório atual como raiz de dados (altere se seus arquivos estiverem em outra pasta)
    DATA_DIR = Path(".")
    
    try:
        # 1. Pipeline de Extração (Extract)
        datasets = load_data(DATA_DIR)
        
        # 2. Pipeline de Transformação (Transform)
        df_ai, ai_revenue, df_gpu_prices, df_fin_gpus, fab_cap, sanctions = preprocess_data(datasets)
        
        # 3. Análise Descritiva
        generate_descriptive_stats(df_ai, df_fin_gpus)
        
        # 4. Visualização de Dados (Load/Visualize)
        plot_dashboards(ai_revenue, df_fin_gpus, df_gpu_prices, fab_cap, sanctions)
        
        logging.info("Pipeline executado com sucesso!")
        
    except Exception as e:
        logging.critical(f"A execução do pipeline falhou: {e}")

# Padrão Pythonico para garantir que o script só rode se for executado diretamente
if __name__ == "__main__":
    main()

2026-06-27 16:45:15,143 - ERROR - Arquivo não encontrado: src\chip_prices.csv. Verifique o diretório.
2026-06-27 16:45:15,144 - CRITICAL - A execução do pipeline falhou: [Errno 2] No such file or directory: 'src\\chip_prices.csv'
